<a href="https://colab.research.google.com/github/shay2222/gpu-ops-toolkit/blob/main/rollout/02_rollout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Tue Oct 20 22:47:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile rollout.py
import datetime, os, time, torch
from transformers import pipeline

LOG = "results/rollout.log"

def log(msg):
    os.makedirs("results", exist_ok=True)
    line = f"{datetime.datetime.utcnow().isoformat()}Z {msg}"
    print(line)
    with open(LOG, "a") as f:
        f.write(line + "\n")

def load(model_id):
    return pipeline("text-generation", model=model_id,
                    torch_dtype=torch.float16, device=0)

def smoke_test(pipe):
    start = time.time()
    out = pipe("Say hello in one word.", max_new_tokens=8, do_sample=False)
    text = out[0]["generated_text"]
    ms = (time.time() - start) * 1000
    ok = len(text.strip()) > 0 and ms < 30000
    return ok, ms, text

def rollout(new_id, previous_id):
    """Returns (pipe, active_model_id, outcome)."""
    log(f"deploy start: {new_id} (previous: {previous_id})")
    pipe = None
    try:
        pipe = load(new_id)
        ok, ms, _ = smoke_test(pipe)
        log(f"smoke test ok={ok} latency_ms={ms:.0f}")
        if ok:
            log(f"deployed {new_id}")
            return pipe, new_id, "deployed"
        raise RuntimeError("smoke test failed")
    except Exception as e:
        log(f"rolling back to {previous_id}: {e}")
        del pipe
        torch.cuda.empty_cache()
        pipe = load(previous_id)
        ok, ms, _ = smoke_test(pipe)
        log(f"rollback smoke test ok={ok} latency_ms={ms:.0f}")
        return pipe, previous_id, "rolled_back"

Writing rollout.py


In [3]:
!pip install -q transformers accelerate
import rollout
pipe, active, outcome = rollout.rollout(
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-0.5B-Instruct")
print("outcome:", outcome, "| active:", active)

2026-10-20T22:48:48.797742Z deploy start: Qwen/Qwen2.5-0.5B-Instruct (previous: Qwen/Qwen2.5-0.5B-Instruct)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

2026-10-20T22:49:06.655864Z smoke test ok=True latency_ms=1869
2026-10-20T22:49:06.656402Z deployed Qwen/Qwen2.5-0.5B-Instruct
outcome: deployed | active: Qwen/Qwen2.5-0.5B-Instruct


rollout path

In [4]:
pipe, active, outcome = rollout.rollout(
    "this-model-does-not-exist",
    "Qwen/Qwen2.5-0.5B-Instruct")
print("outcome:", outcome, "| active:", active)

2026-10-20T22:50:45.789780Z deploy start: this-model-does-not-exist (previous: Qwen/Qwen2.5-0.5B-Instruct)
2026-10-20T22:50:45.990083Z rolling back to Qwen/Qwen2.5-0.5B-Instruct: this-model-does-not-exist is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

2026-10-20T22:50:48.256960Z rollback smoke test ok=True latency_ms=273
outcome: rolled_back | active: Qwen/Qwen2.5-0.5B-Instruct
